## asserted context

This notebook introduces the `asserted_context` judgment record; after running it you can declare and inspect the assumptions that frame an engineering requirement.

The model now has a requirement (`TimelyToast`) and two design variants (`nominal` and `slow`). Before asking whether either variant satisfies the requirement, we need to declare the context: what do we assume about the operating environment? An `asserted_context` record (Hawkins 2011 §3.2) documents one such assumption. The context record does not claim the design is correct — it claims the assumption is appropriate for the evaluation we are about to perform.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """
package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }

    part def Heater {
        attribute power : Real default = 800.0;
    }

    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;

    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }

    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }

    part nominal : Toaster;
    part slow : Toaster {
        attribute :>> cycleTime = 200.0;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

In [ ]:
# Negative control: a constraint that references an attribute not in scope
# confirms that the model enforces referential integrity in constraints.
bad_source = """
package Bad {
    private import ScalarValues::*;
    part def Toaster { attribute cycleTime : Real default = 120.0; }
    requirement def BadReq {
        subject t : Toaster;
        require constraint { t.nonExistentAttr <= 180.0 }
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
from toaster.evidence import ReviewRecord, hash_content, validate_record

context_record = ReviewRecord(
    identifier="AC-001",
    kind="asserted_context",
    claim="120 seconds is the nominal cycle time for standard sliced bread.",
    model_ref="ToasterDemo::nominal",
    content_hash=hash_content(source),
    scope="ToasterDemo",
    criteria="attribute cycleTime : Real default = 120.0",
    premises=[],
    assumption_refs=[],
    evidence_refs=["ToasterDemo::Toaster::cycleTime default = 120.0"],
    rationale="120s is consistent with manufacturer guidance for domestic sliced bread.",
    counterevidence="Thick-cut and frozen bread may require 180-240s.",
    residual_uncertainties="User preference variation not modeled.",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(context_record)
print(f"Record: {context_record.identifier} | kind: {context_record.kind}")
print(f"Claim: {context_record.claim}")
print(f"Validation errors: {errors}")
conn.close()

The `asserted_context` record is the A-F declaration of an assumption; `validate_record()` checks required fields are present (O-S); the printed record shows the claim, rationale, and counterevidence are all populated (E).

Try the chapter exercise in `exercises/ch02/exercise.ipynb`: write an `asserted_context` record for the `brewTemp` assumption in your coffee maker model.